# Rewriting Biographies

Instead of replacing entities with tokens, rewrite mode generates a
privacy-safe paraphrase of the entire text. The pipeline:

1. Detects entities (same as replace mode)
2. Classifies the domain and assigns sensitivity dispositions
3. Generates a rewritten version that obscures sensitive entities
4. Evaluates quality (utility) and privacy (leakage) with an automated repair loop
5. Runs a final LLM judge for informational scores

The result includes **utility_score**, **leakage_mass**, and a
**needs_human_review** flag so you can triage records that need attention.

## Setup

In [1]:
from anonymizer import Anonymizer, AnonymizerConfig, AnonymizerInput, LoggingConfig, Rewrite, configure_logging
from anonymizer.config.rewrite import EvaluationCriteria, PrivacyGoal, RiskTolerance

configure_logging(LoggingConfig.debug())

In [2]:
anonymizer = Anonymizer()

[15:22:39] [INFO] 🔧 Anonymizer initialized with 3 model configs


[15:22:39] [INFO]   |-- 🔎 detector:  gliner-pii-detector


[15:22:39] [INFO]   |-- ✅ validator: gpt-oss-120b


[15:22:39] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


[15:22:39] [DEBUG] NDD adapter: artifact_path=.anonymizer-artifacts


## Input data

In [3]:
input_data = AnonymizerInput(
    source="../data/NVIDIA_synthetic_biographies.csv",
    text_column="biography",
    data_summary="Biographical profiles",
)

## Configure

Spell out what to protect and what to preserve. This gives the rewriter
clear guidance for your domain.

In [4]:
config = AnonymizerConfig(
    rewrite=Rewrite(
        privacy_goal=PrivacyGoal(
            protect="All direct identifiers and quasi-identifier combinations (names, locations, employers, dates)",
            preserve="Career trajectory, educational background, and professional accomplishments",
        ),
        evaluation=EvaluationCriteria(
            risk_tolerance=RiskTolerance.strict,
            max_repair_iterations=3,
        ),
    ),
)

## Preview

In [5]:
preview = anonymizer.preview(
    config=config,
    data=input_data,
    num_records=3,
)

preview.display_record(0)

[15:22:39] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[15:22:39] [DEBUG] input text lengths: min=934, max=1331, mean=1137 chars (25 records)


[15:22:39] [DEBUG] detection config: threshold=0.30, labels=(default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[15:22:39] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[15:22:39] [INFO] 🔍 Running entity detection on 3 records


...

[15:28:05] [INFO] ✅ Preview complete!


[15:28:05] [DEBUG] NDD workflow 'rewrite-final-judge' returned 3 records


[15:28:05] [INFO]   |-- 📋 Rewrite complete (0 failed) [240.6s]


[15:28:05] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Entity,Label,Sensitivity,Protection
40,age,medium,generalize
Bobby,first_name,high,replace
Mexican,race_ethnicity,medium,generalize
veterinarian,occupation,low,left_as_is
Denver,city,medium,generalize
Colorado,state,low,left_as_is
Jefferson High,school_name,medium,generalize
DVM,education_level,medium,left_as_is
University of Colorado Boulder,company_name,medium,generalize
English,language,low,left_as_is


In [6]:
preview.display_record(1)

Entity,Label,Sensitivity,Protection
Idilio,first_name,high,replace
Bell,last_name,high,replace
37,age,medium,generalize
astronomer,occupation,low,left_as_is
Edison,city,medium,generalize
New Jersey,state,medium,generalize
"November 21, 1988",date_of_birth,medium,generalize
Italian,race_ethnicity,low,left_as_is
English,language,low,left_as_is
bachelor’s degree,education_level,low,left_as_is


## Full run

`result.dataframe` has user-facing columns only.
`result.trace_dataframe` has every intermediate column for debugging.

In [7]:
result = anonymizer.run(config=config, data=input_data)

print(result)
result.dataframe.head()

[15:28:05] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[15:28:05] [DEBUG] input text lengths: min=934, max=1331, mean=1137 chars (25 records)


[15:28:05] [DEBUG] detection config: threshold=0.30, labels=(default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[15:28:05] [INFO] 🔍 Running entity detection on 25 records


[15:28:05] [DEBUG] detection aliases: detector=gliner-pii-detector, validator=gpt-oss-120b, augmenter=gpt-oss-120b


...

[15:51:54] [DEBUG]   14304213725050faa50d174576d500b8 (entity-detection: Record missing from workflow output)


[15:51:54] [DEBUG]   9a1de4a9876c5d6f83ececb70bcbfc04 (entity-detection: Record missing from workflow output)


[15:51:54] [INFO] 🎉 Pipeline complete — 25 records processed, 2 total failures


AnonymizerResult(rows=23, columns=6, trace_columns=45, failed_records=2)


,biography,biography_rewritten,utility_score,leakage_mass,any_high_leaked,needs_human_review
0,"Bobby Watford, a 40‑year‑old Mexican veterinar...","Ethan Kline, a Mexican veterinarian in his 40s...",0.861538,0.0,False,False
1,Idilio Bell is a 37‑year‑old astronomer living...,Dario Hawthorne is an astronomer in his late 3...,0.871429,1.8,False,False
2,"Jodi Allison, 36, lives at 204 Bluegrass in Cl...","Leah Harper, in her late 30s, lives at 317 Map...",0.908333,0.0,False,False
3,James Mills is a 69‑year‑old paramedic who liv...,Samuel Bennett is in his late 60s and works as...,0.681818,0.0,False,False
4,Nancy Burton is a 21‑year‑old cashier who live...,Claire Hawkins is in her early twenties and wo...,0.9,0.0,False,False


In [8]:
result.dataframe[["biography_rewritten", "utility_score", "leakage_mass", "needs_human_review"]].head()

,biography_rewritten,utility_score,leakage_mass,needs_human_review
0,"Ethan Kline, a Mexican veterinarian in his 40s...",0.861538,0.0,False
1,Dario Hawthorne is an astronomer in his late 3...,0.871429,1.8,False
2,"Leah Harper, in her late 30s, lives at 317 Map...",0.908333,0.0,False
3,Samuel Bennett is in his late 60s and works as...,0.681818,0.0,False
4,Claire Hawkins is in her early twenties and wo...,0.9,0.0,False


In [9]:
result.trace_dataframe.columns.tolist()

['biography',
 '_anonymizer_record_id',
 '_raw_detected_entities',
 '_seed_entities',
 '_initial_tagged_text',
 '_seed_entities_json',
 '_tag_notation',
 '_merged_tagged_text',
 '_validation_candidates',
 '_validated_entities',
 '_augmented_entities',
 '_merged_entities',
 '_detected_entities',
 'biography_with_spans',
 '_latent_entities',
 'final_entities',
 '_entities_by_value',
 '_entity_examples',
 '_entities_for_replace',
 '_entities_for_replace_json',
 '_replacement_map',
 '_domain',
 '_domain_supplement',
 '_sensitivity_disposition',
 '_sensitivity_disposition_block',
 '_privacy_qa',
 '_rewrite_disposition_block',
 '_meaning_units',
 '_replacement_map_for_prompt',
 '_meaning_units_serialized',
 '_full_rewrite',
 '_quality_qa',
 'biography_rewritten',
 '_repair_iterations',
 '_quality_qa_reanswer',
 '_privacy_qa_reanswer',
 '_quality_qa_compare',
 'utility_score',
 'leakage_mass',
 'any_high_leaked',
 '_needs_repair',
 '_leaked_privacy_items',
 '_rewritten_text__next',
 '_judge_e

## Filter by review flag

Records where automated metrics exceed thresholds are flagged for manual review.

In [10]:
df = result.dataframe
flagged = df[df["needs_human_review"] == True]  # noqa: E712
print(f"{len(flagged)} of {len(df)} records flagged for human review")
flagged.head()

1 of 23 records flagged for human review


,biography,biography_rewritten,utility_score,leakage_mass,any_high_leaked,needs_human_review
6,"Oscar Valenzano, a 52‑year‑old film director l...","Javier Ramos, a film director in his early 50s...",0.87,1.3,True,True
